# **Interactive exploration of Antarctic freshwater fluxes by ocean sector**
*Comparison of low- and high-emissions projections from 1990 to 2300*


For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/1A6_ADTSduj_tZVy_6ob_d2Hpiu8Cgnwz)

*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Interactive exploration of Antarctic freshwater fluxes by ocean sector](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2FSSP126vsSSP585_OceanSectors.ipynb)

**Scientific context**

Freshwater released from the Antarctic Ice Sheet enters the Southern Ocean through several processes, including surface runoff, basal melting beneath ice shelves, and iceberg calving.

Changes in these freshwater inputs can affect:

- Southern Ocean stratification;
- deep-water formation;
- ocean circulation;
- exchanges between the ocean and the Antarctic cryosphere;
- future sea-level and climate projections.

Comparing low- and high-emissions scenarios helps illustrate how these processes may evolve under different future climate pathways.




**Notebook objectives**

This notebook enables users to:

- compare SSP126 and SSP585 projections;
- select one of the Antarctic ocean sectors;
- explore four freshwater-flux components;
- display the median projection and uncertainty interval;
- adjust the time range between 1990 and 2300.

**Data source**

The data are retrieved directly from the **OCEAN:ICE ERDDAP server**:

- **Low-emission scenario** (SSP126 dataset): https://er1.s4oceanice.eu/erddap/griddap/SSP126_FWF_1990_2300_OceanSectors
- **High-emission scenario** (SSP585 dataset): https://er1.s4oceanice.eu/erddap/griddap/SSP585_FWF_1990_2300_OceanSectors

The datasets cover five Antarctic ocean sectors and the period from 1990 to 2300 and provide projections for five Antarctic ocean sectors:

1. Weddell Sea;
2. Indian Ocean;
3. Western Pacific Ocean;
4. Ross Sea;
5. Amundsen and Bellingshausen Sea.

Four freshwater-flux variables are included:

- `Net_Mass_Balance`;
- `Surface_Meltwater_Runoff_Fluxes`;
- `Sub_Shelf_Melt_Fluxes`;
- `Calving_Fluxes`.

Each projection is available for three quantiles:

- `0.05`: lower uncertainty bound;
- `0.50`: median estimate;
- `0.95`: upper uncertainty bound.


**How to use this notebook**

1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.


**Data retrieval, preparation, and interactive time series**

The code below downloads both scenarios, removes the ERDDAP units row, converts relevant columns to numerical values, creates the sector, variable, and time-range controls, and redraws the comparison plot whenever a selection changes.


**Interactive visualization**

The code below creates an interactive viewer that allows the user to select:

- an Antarctic ocean sector;
- a freshwater-flux variable;
- a time interval between 1990 and 2300.

For each emissions scenario, the graph shows:

- the **median projection**, corresponding to the `0.50` quantile;
- the **projection range**, derived from the minimum and maximum values available for each year.

The shaded areas represent the spread of the projections, while the solid lines represent the median values.


In [ ]:
# @title
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Data URLs
url_low = "https://er1.s4oceanice.eu/erddap/griddap/SSP126_FWF_1990_2300_OceanSectors.csv?Net_Mass_Balance%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D,Surface_Meltwater_Runoff_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D,Sub_Shelf_Melt_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D,Calving_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D"
url_high = "https://er1.s4oceanice.eu/erddap/griddap/SSP585_FWF_1990_2300_OceanSectors.csv?Net_Mass_Balance%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D,Surface_Meltwater_Runoff_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D,Sub_Shelf_Melt_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D,Calving_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(5.0)%5D"

def load_data(url):
    df = pd.read_csv(url, skiprows=[1])
    df.columns = [c.strip() for c in df.columns]
    if 'time' not in df.columns:
        time_cols = [c for c in df.columns if 'time' in c.lower()]
        if time_cols: df = df.rename(columns={time_cols[0]: 'time'})
    return df

df_low = load_data(url_low)
df_high = load_data(url_high)

sector_map = {
    "Weddell Sea": 1,
    "Indian Ocean": 2,
    "Western Pacific Ocean": 3,
    "Ross Sea": 4,
    "Amundsen & Bellingshausen Sea": 5
}

columns_to_plot = ['Net_Mass_Balance', 'Surface_Meltwater_Runoff_Fluxes', 'Sub_Shelf_Melt_Fluxes', 'Calving_Fluxes']

sector_dropdown = widgets.Dropdown(options=sector_map, value=1, description='Sector:')
variable_dropdown = widgets.Dropdown(options=columns_to_plot, value=columns_to_plot[0], description='Variable:')
time_slider = widgets.IntRangeSlider(value=[1990, 2300], min=1990, max=2300, step=1, description='Years:', layout={'width': '500px'})
output = widgets.Output()

def update_plot(change):
    with output:
        clear_output(wait=True)
        sid, var = sector_dropdown.value, variable_dropdown.value
        tmin, tmax = time_slider.value
        sector_name = [n for n, v in sector_map.items() if v == sid][0]

        def get_plot_elements(df):
            filtered = df[(df['sector'] == sid) & (df['time'] >= tmin) & (df['time'] <= tmax)]
            median = filtered[filtered['quantile'] == 0.50].sort_values('time')
            lower = filtered.groupby('time')[var].min()
            upper = filtered.groupby('time')[var].max()
            return median, lower, upper

        m_low, l_low, u_low = get_plot_elements(df_low)
        m_high, l_high, u_high = get_plot_elements(df_high)

        plt.figure(figsize=(12, 7))

        # Plot Low Emissions (SSP126)
        plt.fill_between(l_low.index, l_low, u_low, color='blue', alpha=0.15, label='Low Emissions (SSP126) Range')
        plt.plot(m_low['time'], m_low[var], color='blue', linewidth=2, label='Low Emissions (SSP126) Median')

        # Plot High Emissions (SSP585)
        plt.fill_between(l_high.index, l_high, u_high, color='red', alpha=0.15, label='High Emissions (SSP585) Range')
        plt.plot(m_high['time'], m_high[var], color='red', linewidth=2, label='High Emissions (SSP585) Median')

        plt.title(f"{var} - {sector_name}")
        plt.xlabel("Year")
        plt.ylabel("gt/yr")
        # Move legend to the bottom
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

sector_dropdown.observe(update_plot, names='value')
variable_dropdown.observe(update_plot, names='value')
time_slider.observe(update_plot, names='value')

display(widgets.VBox([sector_dropdown, variable_dropdown, time_slider, output]))
update_plot(None)


**Interpretation guidance**

The solid blue line represents the median projection under the low-emissions scenario, while the solid red line represents the median projection under the high-emissions scenario.

The corresponding shaded areas indicate the range of projected values for each year.

A growing separation between the two scenarios suggests that the selected freshwater-flux component becomes increasingly sensitive to the assumed emissions pathway. Differences among sectors highlight the spatial variability of Antarctic ice-sheet and ice-shelf processes.

The visualization is intended for exploratory comparison and should be interpreted together with the metadata and modelling documentation associated with the source datasets.


**Additional resources and acknowledgement**

Python libraries used in this notebook:

- [pandas](https://pandas.pydata.org/docs/) for data processing
- [matplotlib](https://matplotlib.org/stable/index.html) for plotting
- [ipywidgets](https://ipywidgets.readthedocs.io/) for interactive control.

This work has received funding from the European Union Horizon Europe project **Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN:ICE)** under Grant Agreement No. 101060452. UK partners are funded by UK Research and Innovation under the UK Government's Horizon Europe funding guarantee.


<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 80px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
  </div>
</center>